# Suite IDN — Identity: ownership and paths

A reference object's identity is its path: the route from a singleton to it through the relations that own it
(docs/OBJECT_IDENTITY.md). An adjacency marked `owner()` points back to the object's owner, through a relation whose
clauses make the route unambiguous; a key property may carry a naming pattern; a path is a nested value of
`Paths.OfRoot`, read off an object's ownership entries on demand. Step 4a of the store overhaul: the vocabulary, before
objects are identified by their paths.

In [ ]:
from mbse.Schemas.Framework import Builders as B, Paths, Plain, Schemas as S
from support import raises

store = B.Store()
Text = store.Schemas.OfStr.as_schema()
Directory = store.Schemas.OfObject(lambda o: o.name("wts.Directory").ref(True).singleton("wts.Features"))
Definition = store.Schemas.OfObject(lambda o: o.name("wts.Definition").ref(True).properties([
    lambda t: t.name("title").schema([Text]),
]))
Clause = store.Schemas.OfObject(lambda o: o.name("wts.Clause").ref(True))
Defines = store.Schemas.OfRelation(lambda r: (
    r
    .name("wts.Defines")
    .links([
        lambda t: t.name("directory"),
        lambda t: t.name("defn"),
    ])
    .properties([lambda t: t.name("name").schema([Text]).pattern(r"DEF-(\d+\d\d)")])
    .uniques([["directory", "name"], ["defn"]])
))
HasClause = store.Schemas.OfRelation(lambda r: (
    r
    .name("wts.HasClause")
    .links([
        lambda t: t.name("defn"),
        lambda t: t.name("clause"),
    ])
    .properties([lambda t: t.name("name").schema([Text])])
    .uniques([["defn", "name"], ["clause"]])
))
store.Schemas.OfObject(Directory, lambda o: o.adjacencies([lambda t: t.name("dir").relation([Defines]).me("directory")]))
store.Schemas.OfObject(Definition, lambda o: o.adjacencies([
    lambda t: t.name("def").relation([Defines]).me("defn").owner(True),
    lambda t: t.name("clauses").relation([HasClause]).me("defn"),
]))
store.Schemas.OfObject(Clause, lambda o: o.adjacencies([lambda t: t.relation([HasClause]).me("clause").owner(True)]))

## IDN-01 · An adjacency marked `owner()`: its relation has two links, and clauses by which the owned object determines its owner and the owner and keys determine the owned object

In [ ]:
assert Definition.adjacencies["def"].owner is True and Directory.adjacencies["dir"].owner is None
assert [s.validate() for s in (Directory, Definition, Clause, Defines, HasClause)] == [[]] * 5


def owner_of(uniques, links=("up", "down")):
    """An object schema owned through a relation of `links` with `uniques`: its validation."""
    relation = store.Schemas.OfRelation(lambda r: r.links([lambda t, n=n: t.name(n) for n in links]).uniques(uniques))
    return store.Schemas.OfObject(lambda o: o.ref(True).adjacencies([
        lambda t: t.name("up").relation([relation]).me("down").owner(True),
    ])).validate()


assert owner_of([["up"], ["down"]]) == []  # keyless: the owner alone determines the owned object
assert owner_of([["down"]]) == ["adjacency 'up' is its object's owner: no unique clause has 'up' without 'down', so that "
                                "the owned object determines its owner"]
assert owner_of([["up"]]) == ["adjacency 'up' is its object's owner: no unique clause has 'down' without 'up', so that "
                              "the owner and the keys determine the owned object"]
assert owner_of([["up"], ["down"]], ("up", "down", "side")) == [
    "adjacency 'up' is its object's owner: its relation has two links, the owner's and the owned's"]
with raises(TypeError, match="property 'owner' holds bool, got str"):
    store.Schemas.OfObject(lambda o: o.adjacencies([lambda t: t.relation([Defines]).me("defn").owner("yes")]))

## IDN-02 · A key property's naming pattern: the values it gives and reads, and what validation refuses

In [ ]:
name = Defines.entry.properties["name"]
assert name.pattern == r"DEF-(\d+\d\d)" and name.name_for(7) == "DEF-007" and name.name_for(1234) == "DEF-1234"
assert (name.number_of("DEF-693"), name.number_of("DEF-69"), name.number_of("FEA-693")) == (693, None, None)
assert S.OfProperty.Data("k", Text).number_of("DEF-693") is None  # no pattern
bad = store.Schemas.OfRelation(lambda r: (
    r
    .links([lambda t: t.name("a"), lambda t: t.name("b")])
    .properties([
        lambda t: t.name("k").schema([Text]).pattern("DEF-.*"),
        lambda t: t.name("n").schema([store.Schemas.OfInt]).pattern(r"(\d+)"),
    ])
))
assert bad.validate() == [
    "property 'k': naming pattern 'DEF-.*' is not literal text around one group of digits, e.g. 'FEA-(\\d+\\d\\d)'",
    "property 'n': a naming pattern gives text: the property's schema is Schemas.OfStr"]

## IDN-03 · Every store holds the path schemas and `Schemas.HasInstance`; the members of its own relations are owned

In [ ]:
for name in ("Paths.OfRoot", "Paths.OfEntry", "Paths.OfTarget", "Paths.Next"):
    assert store.registered(name).validate() == [] and store.framework(store.registered(name))
assert list(store.Paths.OfRoot.as_schema().properties) == ["singleton", "next"]
assert list(store.Paths.OfEntry.as_schema().properties) == ["i", "o", "properties", "next"]
assert [b.name for b in store.Paths.Next.as_schema().branches] == ["entry", "target"]
other = B.Store()
assert other.Paths.OfRoot.as_schema() is not store.Paths.OfRoot.as_schema()  # each store's own
HasInstance = store.Schemas.HasInstance.as_schema()
assert list(HasInstance.entry.links) == ["schema", "instance"] and list(HasInstance.entry.properties) == ["name"]
assert store.Schemas.OfObject.as_schema().adjacencies["instances"].relation is HasInstance
owned = [(a.relation.name, a.me) for a in store.Schemas.OfProperty.as_schema().anonymous if a.owner]
assert owned == [("Schemas.HasProperty", "property")]
assert all(a.owner for a in store.Schemas.OfObject.as_schema().anonymous if a.relation.name == "Schemas.Defines")

## IDN-04 · An object's path on demand: read off its ownership entries back to a singleton, a nested value; refused where it has no owner, or more than one

In [ ]:
features = store.singleton("wts.Features")
definition = store.wts.Definition(lambda d: d.title("Has 88 keys"))
store.wts.Directory(features, lambda f: f.dir(lambda a: a.insert(lambda e: e.defn(definition).name("DEF-693"))))
clause = store.wts.Clause(lambda c: c)
store.wts.Definition(definition, lambda d: d.clauses(lambda a: a.insert(lambda e: e.clause(clause).name("C-1"))))
to_plain = Plain.ToPlain(store)
path = Paths.of(definition)
assert to_plain(store.Paths.OfRoot.as_schema(), path) == {"singleton": "wts.Features", "next": {"entry": {
    "i": "dir", "o": "def", "properties": {"name": "DEF-693"}, "next": {"target": {}}}}}
assert to_plain(store.Paths.OfRoot.as_schema(), Paths.of(clause)) == {"singleton": "wts.Features", "next": {"entry": {
    "i": "dir", "o": "def", "properties": {"name": "DEF-693"}, "next": {"entry": {
        "i": "clauses", "o": "", "properties": {"name": "C-1"}, "next": {"target": {}}}}}}}  # an anonymous owner
assert to_plain(store.Paths.OfRoot.as_schema(), Paths.of(features)) == {"singleton": "wts.Features",
                                                                         "next": {"target": {}}}
loose = store.wts.Clause(lambda c: c)
with raises(LookupError, match="a 'wts.Clause' has no owner: its path is not known"):
    Paths.of(loose)
Twice = store.Schemas.OfObject(lambda o: o.name("wts.Twice").ref(True))
Holds = store.Schemas.OfRelation(lambda r: r.name("wts.Holds").links([lambda t: t.name("a"), lambda t: t.name("b")])
                                 .uniques([["a"], ["b"]]))
store.Schemas.OfObject(Twice, lambda o: o.adjacencies([
    lambda t: t.name("up").relation([Holds]).me("b").owner(True),
    lambda t: t.name("down").relation([Holds]).me("a"),
]))
first, second = store.wts.Twice(lambda t: t), store.wts.Twice(lambda t: t)
store.wts.Twice(first, lambda t: t.up(lambda a: a.insert(lambda e: e.a(second))))
store.wts.Twice(second, lambda t: t.up(lambda a: a.insert(lambda e: e.a(first))))
with raises(LookupError, match="an ownership cycle: no singleton owns the object"):
    Paths.of(first)
pair = [store.Schemas.OfRelation(lambda r: r.links([lambda t: t.name("a"), lambda t: t.name("b")]).uniques([["a"], ["b"]]))
        for _ in range(2)]
store.Schemas.OfObject(Directory, lambda o: o.adjacencies([lambda t: t.relation([pair[0]]).me("a"),
                                                           lambda t: t.name("rights").relation([pair[1]]).me("a")]))
Split = store.Schemas.OfObject(lambda o: o.name("wts.Split").ref(True).adjacencies([
    lambda t: t.name("left").relation([pair[0]]).me("b").owner(True),
    lambda t: t.name("right").relation([pair[1]]).me("b").owner(True),
]))
split = store.wts.Split(lambda s: s.left(lambda a: a.insert(lambda e: e.a(features))))
assert to_plain(store.Paths.OfRoot.as_schema(), Paths.of(split)) == {"singleton": "wts.Features", "next": {"entry": {
    "i": "", "o": "left", "properties": {}, "next": {"target": {}}}}}  # the owner's adjacency anonymous
store.wts.Split(split, lambda s: s.right(lambda a: a.insert(lambda e: e.a(features))))
with raises(LookupError, match="a 'wts.Split' has more than one owner: its path is not known"):
    Paths.of(split)